# 11 - SigLIP モデル評価（ベースライン）

## 概要
SigLIPモデルをベースラインとして評価する。他のモデルとの比較の基準となる。

## モデル情報
- **Model**: google/siglip-base-patch16-224
- **Embedding dimension**: 768
- **Type**: Multimodal (Image + Text)

## 評価指標
- シルエットスコア (2D/3D) - クラスタ分離度
- Trustworthiness (2D/3D) - 近傍関係の保持度
- カテゴリ内/間距離比 (2D/3D) - 分離の良さ
- PCAの累積寄与率

## 出力
- `data/evaluations/` に評価結果をJSON形式で保存

In [2]:
import time
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm

from image_vector_poc import SigLIPEmbedder
from image_vector_poc.evaluation import EvaluationReporter, evaluate_embeddings

## 設定

In [3]:
# パス設定
DB_PATH = Path("../data/images.duckdb")
OUTPUT_DIR = Path("../data/evaluations")

# バッチサイズ（GPUメモリに応じて調整）
BATCH_SIZE = 32

# 再現性のためのランダムシード
RANDOM_STATE = 42

## 画像カタログの読み込み

In [4]:
# データベースから画像カタログを読み込み
conn = duckdb.connect(str(DB_PATH), read_only=True)

query = """
    SELECT id, file_path, category, file_name
    FROM image_catalog
    ORDER BY category, file_name
"""
catalog = conn.execute(query).fetchall()
conn.close()

# データの整理
image_ids = [r[0] for r in catalog]
file_paths = [r[1] for r in catalog]
categories = [r[2] for r in catalog]
file_names = [r[3] for r in catalog]

# カテゴリ情報
category_labels = np.array(categories)
category_counts = pd.Series(categories).value_counts().to_dict()
unique_categories = list(category_counts.keys())

print(f"Total images: {len(catalog)}")
print(f"Categories: {unique_categories}")
print(f"\nCategory counts:")
for cat, count in category_counts.items():
    print(f"  {cat}: {count}")

Total images: 385
Categories: ['EuroPython2025', 'PyConJP2025', 'PyConJP2025-PreCampHiroshima', 'KashiwaVillagePark2026', 'TokyoNight202505', 'terada']

Category counts:
  EuroPython2025: 129
  PyConJP2025: 77
  PyConJP2025-PreCampHiroshima: 57
  KashiwaVillagePark2026: 56
  TokyoNight202505: 56
  terada: 10


## モデルの初期化

In [5]:
print("Loading SigLIP model...")
embedder = SigLIPEmbedder(device="cuda")
print(f"Model: {embedder.model_name}")
print(f"Embedding dimension: {embedder.embedding_dim}")
print(f"Device: {embedder.device}")

Loading SigLIP model...


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Model: google/siglip-base-patch16-224
Embedding dimension: 768
Device: cuda


## 画像のベクトル化

In [6]:
print(f"Generating embeddings for {len(file_paths)} images...")
print(f"Batch size: {BATCH_SIZE}")

start_time = time.time()
embeddings_list = []

for i in tqdm(range(0, len(file_paths), BATCH_SIZE), desc="Embedding"):
    batch_paths = file_paths[i:i + BATCH_SIZE]
    
    # 画像を読み込み
    batch_images = []
    for path in batch_paths:
        try:
            img = Image.open(path).convert("RGB")
            batch_images.append(img)
        except Exception as e:
            print(f"Error loading {path}: {e}")
            # エラー時はダミー画像を使用
            batch_images.append(Image.new("RGB", (224, 224), color="gray"))
    
    # バッチでベクトル化
    batch_embs = embedder.embed_images(batch_images)
    embeddings_list.append(batch_embs)

embeddings = np.vstack(embeddings_list)
processing_time = time.time() - start_time

print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"Processing time: {processing_time:.2f}s")
print(f"Speed: {len(file_paths) / processing_time:.1f} images/sec")

Generating embeddings for 385 images...
Batch size: 32


Embedding:   0%|          | 0/13 [00:00<?, ?it/s]


Embeddings shape: (385, 768)
Processing time: 86.28s
Speed: 4.5 images/sec


## 評価の実行

In [7]:
print("Running evaluation...")
print("This may take a few minutes (t-SNE computation)...\n")

metrics = evaluate_embeddings(
    embeddings=embeddings,
    labels=category_labels,
    model_name=embedder.model_name,
    embedding_dim=embedder.embedding_dim,
    categories=unique_categories,
    category_counts=category_counts,
    processing_time=processing_time,
    random_state=RANDOM_STATE,
)

Running evaluation...
This may take a few minutes (t-SNE computation)...



## 結果の表示

In [8]:
print("=" * 60)
print("Evaluation Results")
print("=" * 60)
print(f"Model: {metrics.model_name}")
print(f"Embedding dimension: {metrics.embedding_dim}")
print(f"Total images: {metrics.total_images}")
print(f"Processing time: {metrics.processing_time_seconds:.2f}s")

print("\n--- t-SNE Metrics ---")
print(f"\n2D Metrics:")
print(f"  Silhouette score: {metrics.silhouette_2d:.4f}")
print(f"  Trustworthiness:  {metrics.trustworthiness_2d:.4f}")
print(f"  Distance ratio:   {metrics.distance_ratio_2d:.4f}")

print(f"\n3D Metrics:")
print(f"  Silhouette score: {metrics.silhouette_3d:.4f}")
print(f"  Trustworthiness:  {metrics.trustworthiness_3d:.4f}")
print(f"  Distance ratio:   {metrics.distance_ratio_3d:.4f}")

print("\n--- PCA Metrics ---")
print(f"\n2D Metrics:")
print(f"  Silhouette score:  {metrics.pca_silhouette_2d:.4f}")
print(f"  Variance ratio:    {metrics.pca_variance_ratio_2d:.4f}")

print(f"\n3D Metrics:")
print(f"  Silhouette score:  {metrics.pca_silhouette_3d:.4f}")
print(f"  Variance ratio:    {metrics.pca_variance_ratio_3d:.4f}")

Evaluation Results
Model: google/siglip-base-patch16-224
Embedding dimension: 768
Total images: 385
Processing time: 86.28s

--- t-SNE Metrics ---

2D Metrics:
  Silhouette score: 0.1318
  Trustworthiness:  0.9538
  Distance ratio:   0.4867

3D Metrics:
  Silhouette score: 0.1090
  Trustworthiness:  0.9582
  Distance ratio:   0.6269

--- PCA Metrics ---

2D Metrics:
  Silhouette score:  0.0978
  Variance ratio:    0.2418

3D Metrics:
  Silhouette score:  0.1032
  Variance ratio:    0.2946


## 結果の保存

In [9]:
# 評価結果をJSONファイルに保存
reporter = EvaluationReporter(OUTPUT_DIR)
filepath = reporter.save(metrics)
print(f"Results saved to: {filepath}")

Results saved to: ../data/evaluations/google_siglip-base-patch16-224_2026-02-01.json


## GPUメモリのクリーンアップ

In [10]:
# モデルを削除してGPUメモリを解放
del embedder
del embeddings

import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU memory cleared.")
else:
    print("No GPU available.")

GPU memory cleared.


## 指標の解釈

### シルエットスコア
- **範囲**: -1 ~ 1
- **解釈**: 1に近いほどクラスタ分離が良い。0.5以上で良好、0.7以上で優秀。

### Trustworthiness
- **範囲**: 0 ~ 1
- **解釈**: 元の高次元空間の近傍関係がどれだけ保持されているか。0.9以上が望ましい。

### 距離比（Distance Ratio）
- **計算**: 同カテゴリ内平均距離 / 異カテゴリ間平均距離
- **解釈**: 1未満で良好（同カテゴリがより近い）。値が小さいほど分離が良い。

### PCA累積寄与率
- **解釈**: 2D/3Dで元のデータの分散をどれだけ説明できるか。高いほど情報が保持されている。

## SigLIP ベースライン評価結果の分析

### 結果サマリー

| 指標 | 2D | 3D | 評価 |
|------|----|----|------|
| シルエットスコア | 0.132 | 0.109 | やや低い（カテゴリ間の重複あり） |
| Trustworthiness | 0.954 | 0.958 | 優秀（近傍関係を良く保持） |
| 距離比 | 0.487 | 0.627 | 良好（同カテゴリ内が近い） |
| PCA寄与率 | 24.2% | 29.5% | 低い（高次元に情報が分散） |

### 詳細分析

**1. クラスタリング品質（シルエットスコア: 0.13）**
- 0に近い値は、カテゴリ間に自然な重複があることを示す
- イベント写真（EuroPython, PyConJP）は視覚的に類似している可能性が高い
- 風景写真（KashiwaVillagePark, TokyoNight）と人物写真の区別は比較的明確

**2. 構造保持度（Trustworthiness: 0.95）**
- 0.9以上の値は、t-SNEによる次元削減が元の高次元空間の近傍関係を良く保持していることを示す
- SigLIPの埋め込み空間が意味的に一貫していることを示唆

**3. カテゴリ分離度（距離比: 0.49）**
- 1未満の値は、同カテゴリの画像が異カテゴリより近いことを示す
- 約0.5という値は、カテゴリ内の距離が異カテゴリ間の約半分であることを意味
- カテゴリベースの検索に有効

**4. PCA累積寄与率（24%）**
- 768次元のうち2次元で24%しか説明できない
- 情報が高次元に分散しており、単純な線形射影では捉えきれない複雑な構造

### ベースラインとしての評価

SigLIPは以下の特徴を持つベースラインモデルとして機能する：

- **強み**: 高いTrustworthiness、良好な距離比
- **課題**: シルエットスコアの向上余地あり
- **処理速度**: 4.5 images/sec（RTX 4090）

他のモデルとの比較では、このシルエットスコア（0.13）と距離比（0.49）を基準として評価する。